In [6]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project Root:", PROJECT_ROOT)

Project Root: /Users/dhairyas87/Documents/Projects/explainable-credit-risk-management/xai-credit-risk-freddie-mac


In [16]:
import importlib

import scripts.build_target_dataset
import scripts.pipeline
import scripts.build_feature_store

importlib.reload(scripts.build_target_dataset)
importlib.reload(scripts.build_feature_store)
importlib.reload(scripts.pipeline)

<module 'scripts.pipeline' from '/Users/dhairyas87/Documents/Projects/explainable-credit-risk-management/xai-credit-risk-freddie-mac/scripts/pipeline.py'>

In [8]:
import scripts.build_target_dataset as btd

dir(btd)

['Path',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__',
 'build_target_dataset',
 'create_bssi',
 'create_loan_level_performance',
 'create_stress_flag',
 'load_origination',
 'load_performance',
 'validate_loan_counts']

In [9]:
from scripts.pipeline import run_pipeline

QUARTERS = [
    "2018Q1",
    "2018Q2",
    "2018Q3",
    "2018Q4"
]

for quarter in QUARTERS:

    run_pipeline(
        quarter=quarter
    )

RUNNING PIPELINE : 2018Q1
Target dataset already exists: ../data/processed/loan_perf_2018Q1.parquet
Master dataset already exists: ../data/processed/master_dataset_2018Q1.parquet

Pipeline completed.
RUNNING PIPELINE : 2018Q2
Target dataset already exists: ../data/processed/loan_perf_2018Q2.parquet
Master dataset already exists: ../data/processed/master_dataset_2018Q2.parquet

Pipeline completed.
RUNNING PIPELINE : 2018Q3
Target dataset already exists: ../data/processed/loan_perf_2018Q3.parquet
Master dataset already exists: ../data/processed/master_dataset_2018Q3.parquet

Pipeline completed.
RUNNING PIPELINE : 2018Q4
Target dataset already exists: ../data/processed/loan_perf_2018Q4.parquet
Master dataset already exists: ../data/processed/master_dataset_2018Q4.parquet

Pipeline completed.


In [10]:
from scripts.build_combined_dataset import (
    build_combined_dataset
)

combined_df = build_combined_dataset(
    input_files=[
        "../data/processed/master_dataset_2018Q1.parquet",
        "../data/processed/master_dataset_2018Q2.parquet",
        "../data/processed/master_dataset_2018Q3.parquet",
        "../data/processed/master_dataset_2018Q4.parquet"
    ],
    output_path="../data/processed/master_dataset_2018.parquet"
)

Loading ../data/processed/master_dataset_2018Q1.parquet
Loading ../data/processed/master_dataset_2018Q2.parquet
Loading ../data/processed/master_dataset_2018Q3.parquet
Loading ../data/processed/master_dataset_2018Q4.parquet
Combined Shape: (1285434, 49)
Saved: ../data/processed/master_dataset_2018.parquet


In [17]:
from scripts.build_feature_store import (
    build_feature_store
)

build_feature_store(
    input_path="../data/processed/master_dataset_2018.parquet",
    output_dir="../data/modeling"
)

BUILDING FEATURE STORE

Loading dataset...
Dataset Shape: (1285434, 49)

Creating baseline dataset...
Saved baseline datasets.
Baseline Shape: (1285434, 34)

Creating BSS dataset...
Saved bss datasets.
BSS Shape: (1285434, 37)

Feature store creation complete.


In [22]:
import pandas as pd

pd.read_parquet(
    "../data/modeling/train_baseline.parquet"
).shape



(661318, 34)

In [24]:
import pandas as pd
pd.read_parquet(
    "../data/modeling/train_bss.parquet"
).shape

(661318, 37)

In [25]:
for name in [
    "train_baseline",
    "valid_baseline",
    "test_baseline"
]:

    df = pd.read_parquet(
        f"../data/modeling/{name}.parquet"
    )

    print(
        name,
        df.shape
    )

train_baseline (661318, 34)
valid_baseline (336669, 34)
test_baseline (287447, 34)


In [26]:
for name in [
    "train_baseline",
    "valid_baseline",
    "test_baseline"
]:

    df = pd.read_parquet(
        f"../data/modeling/{name}.parquet"
    )

    print(
        name,
        round(
            df["stress_flag"].mean() * 100,
            2
        )
    )

train_baseline 13.97
valid_baseline 13.74
test_baseline 13.39
